# Reading and writing
or more explicitly:
# Reading and writing datasets and metadata

This tutorial shows how `xsnow` handles data I/O and metadata:

1) Read datasets
2) Inspect dimensions, coordinates, and variables
3) View and edit metadata
4) Write datasets to file

`xsnow` focuses on parsing SNOWPACK (PRO/SMET) and CROCUS (NetCDF) and will provide basic readers for CAAML/JSON. It also supports an efficient xsnow NetCDF format for caching large datasets.

- **SNOWPACK**: native PRO and SMET
- **CROCUS**: NetCDF *(planned)*
- **CAAML**: XML/JSON *(planned)*
- **xsnow NetCDF**: optimized for `xsnow`’s data model `xsnowDataset`

```{admonition} Sample datasets
:class: note

`xsnow` ships several sample datasets. Two lightweight ones are available via `xsnow.single_profile()` and `xsnow.single_profile_timeseries()`. More datasets live in [`xsnow.sample_data`](../api/_generated/xsnow.sample_data) (see API docs).
```

## 1. Reading example SNOWPACK datasets
For this SNOWPACK section, we use a sample dataset of gridded simulations shipped in native SNOWPACK formats. Get the sample data path:

In [1]:
import xsnow
datapath = xsnow.sample_data.snp_gridded_dir()

The function returns a directory path. Its structure looks like:

In [2]:
# cell hidden through metadata
import os
print(f"Data location: {datapath}")
for root, dirs, files in os.walk(datapath):
    level = root.replace(datapath, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    fcounter = 0
    for f in sorted(files):
        if fcounter < 3 or fcounter > len(files)-3:
            print(f"{subindent}{f}")
        elif fcounter == 3:
            print(f"{subindent}...")
        fcounter += 1


Data location: /home/flo/.cache/xsnow-snp-gridded
xsnow-snp-gridded/
    pros/
        gridded/
            VIR1A.pro
            VIR1A.smet
            VIR1A1.pro
            ...
            VIR5A4.pro
            VIR5A4.smet
    smets/
        gridded/
            forecast/
                VIR1A.smet
                VIR2A.smet
                VIR3A.smet
                VIR4A.smet
                VIR5A.smet
            nowcast/
                VIR1A.smet
                VIR2A.smet
                VIR3A.smet
                VIR4A.smet
                VIR5A.smet


### Reading a single file
Read a single **PRO** file:

In [3]:
ds = xsnow.read(f"{datapath}/pros/gridded/VIR1A.pro")
print(ds)

INFO:xsnow.xsnow_io:Loading 1 datasets eagerly with 13 workers...


<xsnowDataset>
  Locations:  1
  Timestamps: 373 (2024-01-18--2024-02-02)
  Profiles:   NotImplementedYet

    employing the <xarray.Dataset> Size: 488kB
    Dimensions:                   (location: 1, time: 373, slope: 1,
                                   realization: 1, layer: 12)
    Coordinates:
      * location                  (location) <U5 20B 'VIR1A'
      * time                      (time) datetime64[ns] 3kB 2024-01-18 ... 2024-0...
      * slope                     (slope) int64 8B 0
      * realization               (realization) int64 8B 0
      * layer                     (layer) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
        latitude                  (location) float64 8B 47.15
        longitude                 (location) float64 8B 11.19
        altitude                  (location) float64 8B 2.372e+03
        inclination               (slope) int64 8B 0
        azimuth                   (slope) int64 8B 0
        z                         (location, time, slope, realizat

The summary shows 1 location and 373 timestamps. `xsnowDataset` wraps an underlying `xarray.Dataset` with additional dimensions (`slope`, `realization`, `layer`).

Read a single **SMET** file (no `layer` dimension):

In [4]:
# Read a single smet file
ds = xsnow.read(f"{datapath}/pros/gridded/VIR1A.smet")
print(ds)

<xsnowDataset>
  Locations:  1
  Timestamps: 416 (2024-01-16--2024-02-02)
  Profiles:   NotImplementedYet

    employing the <xarray.Dataset> Size: 213kB
    Dimensions:               (location: 1, slope: 1, realization: 1, time: 416)
    Coordinates:
      * location              (location) object 8B 'VIR1A'
      * slope                 (slope) int64 8B 0
      * realization           (realization) int64 8B 0
      * time                  (time) datetime64[ns] 3kB 2024-01-16T05:00:00 ... 2...
        latitude              (location) float64 8B 47.15
        longitude             (location) float64 8B 11.19
        altitude              (location) float64 8B 2.372e+03
        inclination           (slope) float64 8B nan
        azimuth               (slope) float64 8B nan
    Data variables: (12/63)
        Qs                    (location, slope, realization, time) float64 3kB -3...
        Ql                    (location, slope, realization, time) float64 3kB -3...
        Qg        

### Reading multiple files
Read multiple files by (1) passing a list of paths or (2) pointing to a directory. Files are merged on shared coordinates (e.g., `location`, `time`). Note that for parsing SNOWPACK formats, the merging is based on identical `StationName`s and `station_name`s (properties of PRO and SMET files). Therefore, please don't use the same station names for different locations.

In [5]:
# Reading a list of filepaths
ds = xsnow.read([
    f"{datapath}/pros/gridded/VIR1A.pro",            # profile data
    f"{datapath}/pros/gridded/VIR1A.smet",           # scalar output data
    f"{datapath}/smets/gridded/nowcast/VIR1A.smet",  # weather input data
])

INFO:xsnow.xsnow_io:Loading 1 datasets eagerly with 13 workers...


In [6]:
# Reading all files within one directory
ds = xsnow.read(f"{datapath}/pros/gridded/")

INFO:xsnow.xsnow_io:Loading 25 datasets lazily using Dask and 13 workers...


In [7]:
print(ds)

<xsnowDataset>
  Locations:  5
  Timestamps: 416 (2024-01-16--2024-02-02)
  Profiles:   NotImplementedYet

    employing the <xarray.Dataset> Size: 42MB
    Dimensions:                   (location: 5, time: 416, slope: 5,
                                   realization: 1, layer: 33)
    Coordinates:
      * location                  (location) <U5 100B 'VIR1A' 'VIR2A' ... 'VIR5A'
      * time                      (time) datetime64[ns] 3kB 2024-01-16T05:00:00 ....
      * realization               (realization) int64 8B 0
      * layer                     (layer) int64 264B 0 1 2 3 4 5 ... 28 29 30 31 32
      * slope                     (slope) int64 40B 0 1 2 3 4
        latitude                  (location) float64 40B 47.15 47.44 ... 47.44 47.37
        longitude                 (location) float64 40B 11.19 11.29 11.3 11.39 11.5
        altitude                  (location) float64 40B 2.372e+03 ... 2.066e+03
        inclination               (slope) int64 40B 0 38 38 38 38
        az

### Adding SMET files to an existing dataset
Add `.smet` files to your `xsnowDataset` after reading it with `xsnowDataset.merge_smet()`. You can pass a single file, a list of files, or a directory.

In [8]:
ds = xsnow.read(f"{datapath}/pros/gridded/VIR1A.pro")
ds = ds.merge_smet(f"{datapath}/pros/gridded/VIR1A.smet")

INFO:xsnow.xsnow_io:Loading 1 datasets eagerly with 13 workers...


```{tip}
Use `var_prefix` to avoid name collisions (e.g., input SMET variables as `input_TA`, `input_PSUM`, etc.).
This keeps data organized and prevents overwriting variables with identical names across input/output SMETs.
```

## 2. Lazy versus eager loading
When reading large datasets into the `xsnow` gridded data structure, it’s important to understand the difference between lazy and eager loading — and how this affects memory usage.

 * <u>Eager loading</u> means: as soon as you call the read function, the **entire dataset is loaded into memory** (i.e., into a concrete `xr.Dataset`). Further operations are applied on that in-memory object.

 * <u>Lazy loading</u> means: the read function returns a deferred object (or a data structure with deferred chunks) that **doesn’t immediately load all data**. You can continue chaining selections/transformations, and only when you trigger a final compute will the data actually be read from disk and loaded into memory.

For more information, see [Handling large datasets](./lazy_analysis). 

## 3. Subsetting data while reading

As a convenience, the dataset can already be subset during the call to `read()`. The entire dataset is being parsed first, and then subset to the desired coordinates. This is particularly powerful if you load your data **lazily** first---*"load only what you need"*. For example:


In [9]:
subset = xsnow.read(f"{datapath}/pros/gridded/", 
                location=['VIR1A', 'VIR3A'],
                slope={'inclination': 38, 'azimuth': 180},
                time=slice('2024-01-16T10:00', '2024-01-16T12:00'),
                lazy=True)

INFO:xsnow.xsnow_io:Loading 25 datasets lazily using Dask and 13 workers...


ValueError: No slope matches criteria: inclination == 38, azimuth == 180

```{warning}

The implementation of this feature is not completely done yet. You may not be able to

 * Subset while reading netcdf files
 * Apply time slice subsetting in `read` yet as it set's inclination and azimuth information to `nan`.
```

## 4. Inspecting structure and metadata
You can use standard xarray accessors to explore shape and metadata:

In [10]:
# List dimensions and sizes
ds.sizes

Frozen({'location': 1, 'time': 416, 'slope': 1, 'realization': 1, 'layer': 12})

In [11]:
# List coordinates and their info
ds.coords

Coordinates:
  * location     (location) <U5 20B 'VIR1A'
  * time         (time) datetime64[ns] 3kB 2024-01-16T05:00:00 ... 2024-02-02...
  * slope        (slope) int64 8B 0
  * realization  (realization) int64 8B 0
  * layer        (layer) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
    latitude     (location) float64 8B 47.15
    longitude    (location) float64 8B 11.19
    altitude     (location) float64 8B 2.372e+03
    inclination  (slope) int64 8B 0
    azimuth      (slope) int64 8B 0
    z            (location, time, slope, realization, layer) float32 20kB nan...

In [12]:
# List all data variables
ds.data_vars

Data variables:
    air_content               (location, time, slope, realization, layer) float32 20kB ...
    bond_size                 (location, time, slope, realization, layer) float32 20kB ...
    coord_number              (location, time, slope, realization, layer) float32 20kB ...
    dendricity                (location, time, slope, realization, layer) float32 20kB ...
    density                   (location, time, slope, realization, layer) float32 20kB ...
    element_ID                (location, time, slope, realization, layer) float32 20kB ...
    grain_size                (location, time, slope, realization, layer) float32 20kB ...
    grain_type                (location, time, slope, realization, layer) float32 20kB ...
    gs_difference             (location, time, slope, realization, layer) float32 20kB ...
    hand_hardness             (location, time, slope, realization, layer) float32 20kB ...
    hardness_difference       (location, time, slope, realization, layer) 

Inspect individual variables (e.g., snow density):

In [13]:
print(ds['density'])

<xarray.DataArray 'density' (location: 1, time: 416, slope: 1, realization: 1,
                             layer: 12)> Size: 20kB
array([[[[[  nan,   nan,   nan, ...,   nan,   nan,   nan]]],


        [[[  nan,   nan,   nan, ...,   nan,   nan,   nan]]],


        [[[  nan,   nan,   nan, ...,   nan,   nan,   nan]]],


        ...,


        [[[213.9, 209. , 223.7, ..., 171.9, 142.4, 124.9]]],


        [[[214. , 209.2, 223.7, ..., 172.6, 143. , 125. ]]],


        [[[214.2, 209.3, 223.8, ..., 173.3, 143.6, 125. ]]]]],
      shape=(1, 416, 1, 1, 12), dtype=float32)
Coordinates:
  * location     (location) <U5 20B 'VIR1A'
  * time         (time) datetime64[ns] 3kB 2024-01-16T05:00:00 ... 2024-02-02...
  * slope        (slope) int64 8B 0
  * realization  (realization) int64 8B 0
  * layer        (layer) int64 96B 0 1 2 3 4 5 6 7 8 9 10 11
    latitude     (location) float64 8B 47.15
    longitude    (location) float64 8B 11.19
    altitude     (location) float64 8B 2.372e+03
    inclinati

Each variable carries **attributes (metadata)** like units and descriptions.

View dataset- and variable-level attributes:

In [14]:
# dataset attributes
ds.attrs

{'Conventions': 'CF-1.8',
 'title': 'Snowpack Profile for VIR1A',
 'source': 'SNOWPACK model output (.pro file)',
 'history': '2025-10-28T09:38:33.068309+00:00: Created by xsnow.',
 'crs': 'EPSG:4326'}

In [15]:
# variable-level attributes
ds['density'].attrs

{'standard_name': 'mass_density_of_snow',
 'units': 'kg m-3',
 'long_name': 'mass density of snow'}

## 5. Adding or modifying metadata

Metadata are stored under `.attrs`. You can freely edit or add entries.

For example, add a comment:

In [16]:
ds.attrs['comments'] = 'You can document your thoughts!'
ds.attrs

{'Conventions': 'CF-1.8',
 'title': 'Snowpack Profile for VIR1A',
 'source': 'SNOWPACK model output (.pro file)',
 'history': '2025-10-28T09:38:33.068309+00:00: Created by xsnow.',
 'crs': 'EPSG:4326',
 'comments': 'You can document your thoughts!'}

Modify variable-level attributes, e.g., change units or add remarks:

In [17]:
ds['density'].attrs['comment'] = "What's the density of ice cream?"
ds['density'].attrs

{'standard_name': 'mass_density_of_snow',
 'units': 'kg m-3',
 'long_name': 'mass density of snow',
 'comment': "What's the density of ice cream?"}

It’s good practice to record processing steps in a `history` attribute:

In [18]:
from datetime import datetime
history_line = f"{datetime.now().isoformat(timespec='seconds')}: modified metadata interactively"
ds.attrs['history'] = ds.attrs.get('history', '') + '\n' + history_line
print(ds.attrs['history'])

2025-10-28T09:38:33.068309+00:00: Created by xsnow.
2025-10-28T10:39:42: modified metadata interactively


## 6. Writing datasets to file

You can save your `xsnowDataset` back to file(s). To learn about supported formats, explore the `to_*()` methods in the [xsnow_io](../api/_generated/xsnow.xsnow_io) module API docs. For example,

- `to_netcdf()`: An almost exact dump of the `xsnowDataset` that can be parsed again by `read()`.
- `to_pro()` and `to_smet()`: For every (location, time, slope, realization) you can write one `.smet` and `.pro` file.

First, create a temporary directory to write to:

In [19]:
import tempfile
tmpdir = tempfile.mkdtemp(prefix="xsnow-demo-")

### Writing to .pro / .smet

In [20]:
# Write a single profile timeseries to .pro/.smet
ds.to_pro(f"{tmpdir}/test.pro")
ds.to_smet(f"{tmpdir}/test.smet")

INFO:root:Dataset successfully saved to /tmp/xsnow-demo-oba8uujz/test.pro
INFO:root:Dataset successfully saved to /tmp/xsnow-demo-oba8uujz/test.smet


### Dumping the entire dataset to NetCDF

In [21]:
ds.to_netcdf(f"{tmpdir}/test.nc")

INFO:root:Dataset successfully saved to /tmp/xsnow-demo-oba8uujz/test.nc
